# Chapter 4.5: Distributed Execution — Ray, Multiprocessing, TP/PP

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** vLLM's distributed architecture and execution models
2. **Compare** Ray-based vs multiprocessing-based distributed execution
3. **Analyze** tensor parallelism (TP): how weights and activations are sharded
4. **Explain** pipeline parallelism (PP) and micro-batch scheduling
5. **Trace** NCCL communication patterns in vLLM
6. **Configure** multi-GPU and multi-node serving
7. **Profile** communication overhead and identify bottlenecks

---

## Prerequisites
- Basic understanding of distributed computing
- Familiarity with GPU programming (CUDA)
- Understanding of transformer architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part4/chapter_4.5_distributed.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part4/chapter_4.5_distributed.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Distributed Architecture Overview

### Why Distributed Execution?

Large language models often exceed the memory of a single GPU:

```
Model Sizes vs GPU Memory:
  LLaMA-7B:   ~14 GB (FP16)  → 1× A100-80GB ✓
  LLaMA-13B:  ~26 GB (FP16)  → 1× A100-80GB ✓
  LLaMA-70B:  ~140 GB (FP16) → 2× A100-80GB (TP=2)
  LLaMA-405B: ~810 GB (FP16) → 16× A100-80GB (TP=8, PP=2)
  GPT-4 (est): ~1.7 TB       → 32+ GPUs
```

### Parallelism Types

```
┌─────────────────────────────────────────────────────────────┐
│                    Parallelism Strategies                    │
├─────────────────────┬──────────────────┬────────────────────┤
│ Tensor Parallelism  │ Pipeline         │ Data Parallelism   │
│ (TP)                │ Parallelism (PP) │ (DP)               │
│                     │                  │                    │
│ Split WITHIN layers │ Split ACROSS     │ Replicate model,   │
│ (weights, activs)   │ layers           │ split data         │
│                     │                  │                    │
│ All-reduce per      │ Send activations │ No communication   │
│ layer               │ between stages   │ during inference   │
│                     │                  │                    │
│ Best within node    │ Best across      │ Best for throughput │
│ (NVLink)            │ nodes            │ (independent)      │
└─────────────────────┴──────────────────┴────────────────────┘
```

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import List, Dict, Tuple, Optional
import time

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")

## 2. vLLM's Distributed Architecture

```
vLLM Distributed Architecture:

┌────────────────────────────────────────────────────────┐
│                     LLMEngine                          │
│  ┌──────────┐  ┌──────────────┐  ┌──────────────────┐ │
│  │ Scheduler │  │ TokenizerGrp │  │ OutputProcessor  │ │
│  └──────────┘  └──────────────┘  └──────────────────┘ │
│                         │                              │
│              ┌──────────┴──────────┐                   │
│              │  ExecutorBase       │                   │
│              │  (Ray or MP)        │                   │
│              └──────────┬──────────┘                   │
└─────────────────────────┼──────────────────────────────┘
                          │
         ┌────────────────┼────────────────┐
         │                │                │
    ┌────▼────┐     ┌────▼────┐     ┌────▼────┐
    │ Worker 0 │     │ Worker 1 │     │ Worker 2 │
    │ (GPU 0)  │     │ (GPU 1)  │     │ (GPU 2)  │
    │          │     │          │     │          │
    │┌────────┐│     │┌────────┐│     │┌────────┐│
    ││ModelRun││     ││ModelRun││     ││ModelRun││
    ││CacheEng││     ││CacheEng││     ││CacheEng││
    │└────────┘│     │└────────┘│     │└────────┘│
    └──────────┘     └──────────┘     └──────────┘
         │                │                │
         └────── NCCL ────┴────── NCCL ────┘
```

### Source Code Structure

```
vllm/executor/
├── executor_base.py       # Abstract executor interface
├── gpu_executor.py        # Single-GPU executor
├── ray_gpu_executor.py    # Ray-based multi-GPU executor
├── mp_gpu_executor.py     # Multiprocessing-based multi-GPU executor
└── ray_utils.py           # Ray utilities

vllm/worker/
├── worker.py              # GPU worker (runs on each GPU)
├── model_runner.py        # Model execution logic
└── cache_engine.py        # KV-cache management per GPU

vllm/distributed/
├── communication_op.py    # NCCL wrappers
├── parallel_state.py      # TP/PP group management
└── device_communicators/  # Backend-specific communicators
```

## 3. Ray-Based vs Multiprocessing-Based Execution

### Ray-Based (`RayGPUExecutor`)

```python
# From vllm/executor/ray_gpu_executor.py (simplified)
class RayGPUExecutor(ExecutorBase):
    def _init_workers_ray(self):
        # Create Ray actors for each GPU
        self.workers = []
        for rank in range(self.parallel_config.world_size):
            worker = ray.remote(
                num_gpus=1,
            )(RayWorkerWrapper).remote(
                worker_module_name="vllm.worker.worker",
                worker_class_name="Worker",
            )
            self.workers.append(worker)
        
        # Initialize NCCL groups
        self._run_workers("init_device")
        self._run_workers("load_model")
    
    def execute_model(self, execute_model_req):
        # Broadcast inputs to all workers
        # Each worker runs the same model with TP sharding
        all_outputs = self._run_workers(
            "execute_model",
            execute_model_req=execute_model_req,
        )
        return all_outputs[0]  # Only rank 0 output matters
```

### Multiprocessing-Based (`MultiprocessingGPUExecutor`)

```python
# From vllm/executor/mp_gpu_executor.py (simplified)
class MultiprocessingGPUExecutor(ExecutorBase):
    def _init_workers(self):
        # Use Python multiprocessing instead of Ray
        # Avoids Ray dependency and overhead
        for rank in range(self.parallel_config.world_size):
            process = multiprocessing.Process(
                target=_worker_main,
                args=(rank, ...),
            )
            process.start()
            self.workers.append(process)
```

### Comparison

| Feature | Ray | Multiprocessing |
|---------|-----|----------------|
| Multi-node support | Yes | No |
| Fault tolerance | Yes (actor restart) | No |
| Overhead | Higher (Ray runtime) | Lower |
| Dependency | Requires Ray | Python stdlib |
| Best for | Production, multi-node | Development, single-node |

In [ ]:
class DistributedExecutorSimulator:
    """
    Simulates vLLM's distributed execution to demonstrate
    communication patterns and parallelism strategies.
    """
    
    def __init__(self, tp_size: int = 1, pp_size: int = 1):
        self.tp_size = tp_size
        self.pp_size = pp_size
        self.world_size = tp_size * pp_size
        
        # Communication log
        self.comm_log: List[dict] = []
        self.total_comm_bytes = 0
    
    def log_comm(self, op_type: str, src: int, dst: int, 
                  bytes_transferred: int, description: str):
        entry = {
            'op': op_type,
            'src': src,
            'dst': dst,
            'bytes': bytes_transferred,
            'desc': description,
        }
        self.comm_log.append(entry)
        self.total_comm_bytes += bytes_transferred
    
    def simulate_tp_layer(
        self, 
        batch_size: int,
        hidden_size: int,
        dtype_bytes: int = 2,
    ):
        """Simulate one transformer layer with tensor parallelism."""
        if self.tp_size == 1:
            return
        
        # All-reduce after attention output projection
        attn_output_bytes = batch_size * hidden_size * dtype_bytes
        # Ring all-reduce: 2 * (tp_size-1)/tp_size * data_size
        comm_bytes = int(2 * (self.tp_size - 1) / self.tp_size * attn_output_bytes)
        for rank in range(self.tp_size):
            self.log_comm('all_reduce', rank, -1, comm_bytes,
                         f'Attention output all-reduce (rank {rank})')
        
        # All-reduce after MLP down projection  
        mlp_output_bytes = batch_size * hidden_size * dtype_bytes
        comm_bytes = int(2 * (self.tp_size - 1) / self.tp_size * mlp_output_bytes)
        for rank in range(self.tp_size):
            self.log_comm('all_reduce', rank, -1, comm_bytes,
                         f'MLP output all-reduce (rank {rank})')
    
    def simulate_pp_transfer(
        self,
        batch_size: int,
        hidden_size: int,
        src_stage: int,
        dst_stage: int,
        dtype_bytes: int = 2,
    ):
        """Simulate pipeline stage transfer."""
        transfer_bytes = batch_size * hidden_size * dtype_bytes
        self.log_comm('p2p_send', src_stage, dst_stage, transfer_bytes,
                     f'PP transfer stage {src_stage} -> {dst_stage}')
    
    def simulate_forward(
        self,
        batch_size: int,
        num_layers: int,
        hidden_size: int,
    ) -> dict:
        """Simulate a full forward pass."""
        self.comm_log = []
        self.total_comm_bytes = 0
        
        layers_per_stage = num_layers // self.pp_size
        
        for stage in range(self.pp_size):
            for layer in range(layers_per_stage):
                self.simulate_tp_layer(batch_size, hidden_size)
            
            # Pipeline transfer between stages
            if stage < self.pp_size - 1:
                self.simulate_pp_transfer(
                    batch_size, hidden_size, stage, stage + 1)
        
        return {
            'total_comm_bytes': self.total_comm_bytes,
            'num_all_reduces': sum(1 for e in self.comm_log if e['op'] == 'all_reduce'),
            'num_p2p_sends': sum(1 for e in self.comm_log if e['op'] == 'p2p_send'),
        }

# Example: 70B model on 8 GPUs with TP=4, PP=2
sim = DistributedExecutorSimulator(tp_size=4, pp_size=2)
result = sim.simulate_forward(batch_size=32, num_layers=80, hidden_size=8192)

print("=== Distributed Forward Pass (TP=4, PP=2, 80 layers) ===")
print(f"Total communication: {result['total_comm_bytes'] / 1024 / 1024:.1f} MB")
print(f"All-reduce operations: {result['num_all_reduces']}")
print(f"P2P send operations: {result['num_p2p_sends']}")

## 4. Tensor Parallelism (TP) Deep Dive

### How TP Shards Weights

```
Column Parallelism (QKV projections, gate/up projections):
  Full weight: W [hidden, output_dim]
  Shard along output_dim: W_i [hidden, output_dim/TP]
  
  GPU 0: W[:, 0:d/4]    → partial output[:, 0:d/4]
  GPU 1: W[:, d/4:d/2]  → partial output[:, d/4:d/2]
  GPU 2: W[:, d/2:3d/4] → partial output[:, d/2:3d/4]
  GPU 3: W[:, 3d/4:d]   → partial output[:, 3d/4:d]

Row Parallelism (output projection, down projection):
  Full weight: W [input_dim, hidden]
  Shard along input_dim: W_i [input_dim/TP, hidden]
  
  GPU 0: W[0:d/4, :]    × partial_input → partial_output
  GPU 1: W[d/4:d/2, :]  × partial_input → partial_output
  GPU 2: W[d/2:3d/4, :] × partial_input → partial_output  
  GPU 3: W[3d/4:d, :]   × partial_input → partial_output
  
  All-reduce(partial_outputs) → full output
```

### Transformer Layer with TP=4

```
Input: x [batch, hidden]  (replicated on all GPUs)
                │
    ┌───────────┼───────────┐
    │    Attention Block     │
    │                       │
    │  QKV = x @ W_qkv     │  Column parallel: each GPU gets partial QKV
    │  attn = Attention(QKV)│  Each GPU: full attention on its heads
    │  out = attn @ W_o     │  Row parallel: each GPU has partial result
    │  ──── ALL-REDUCE ──── │  ★ Communication point 1
    │  x = x + out          │
    │                       │
    ├───────────────────────┤
    │       MLP Block       │
    │                       │
    │  gate = x @ W_gate    │  Column parallel
    │  up = x @ W_up        │  Column parallel
    │  h = SiLU(gate) * up  │  Local computation
    │  out = h @ W_down     │  Row parallel
    │  ──── ALL-REDUCE ──── │  ★ Communication point 2
    │  x = x + out          │
    └───────────────────────┘
                │
Output: x [batch, hidden]  (replicated on all GPUs)
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Figure 1: Tensor Parallelism — Single Transformer Layer Split Across 4 GPUs ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')
fig.patch.set_facecolor('white')

gpu_colors = [BLUE, GREEN, ORANGE, PURPLE]
gpu_xs = [2, 5.5, 9, 12.5]

# GPU labels
for i, (x, color) in enumerate(zip(gpu_xs, gpu_colors)):
    ax.text(x+0.75, 9.5, f'GPU {i}', fontsize=11, fontweight='bold', ha='center', color=color)

# Input (replicated)
for i, x in enumerate(gpu_xs):
    rect = mpatches.FancyBboxPatch((x, 8.5), 1.5, 0.7, boxstyle="round,pad=0.05",
        facecolor='#95A5A6', alpha=0.7, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+0.75, 8.85, 'x (full)', fontsize=7, ha='center', color='white', fontweight='bold')

# Column-parallel QKV
for i, (x, color) in enumerate(zip(gpu_xs, gpu_colors)):
    rect = mpatches.FancyBboxPatch((x, 7.2), 1.5, 0.8, boxstyle="round,pad=0.05",
        facecolor=color, alpha=0.85, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+0.75, 7.6, f'QKV\nheads {i*16}-{i*16+15}', fontsize=6, ha='center',
            color='white', fontweight='bold')
    ax.annotate('', xy=(x+0.75, 8.0), xytext=(x+0.75, 8.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.2))
ax.text(0.3, 7.6, 'Column\nParallel', fontsize=8, fontweight='bold', color='#555')

# Attention (local per GPU)
for i, (x, color) in enumerate(zip(gpu_xs, gpu_colors)):
    rect = mpatches.FancyBboxPatch((x, 6.0), 1.5, 0.8, boxstyle="round,pad=0.05",
        facecolor=color, alpha=0.85, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+0.75, 6.4, f'Attn\n(local)', fontsize=7, ha='center', color='white', fontweight='bold')
    ax.annotate('', xy=(x+0.75, 6.8), xytext=(x+0.75, 7.2),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.2))

# Row-parallel output proj
for i, (x, color) in enumerate(zip(gpu_xs, gpu_colors)):
    rect = mpatches.FancyBboxPatch((x, 4.8), 1.5, 0.8, boxstyle="round,pad=0.05",
        facecolor=color, alpha=0.85, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+0.75, 5.2, f'OutProj\n(partial)', fontsize=7, ha='center', color='white', fontweight='bold')
    ax.annotate('', xy=(x+0.75, 5.6), xytext=(x+0.75, 6.0),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.2))
ax.text(0.3, 5.2, 'Row\nParallel', fontsize=8, fontweight='bold', color='#555')

# ALL-REDUCE bar
rect = mpatches.FancyBboxPatch((1.5, 3.8), 13.5, 0.7, boxstyle="round,pad=0.08",
    facecolor=RED, alpha=0.9, edgecolor='white', linewidth=2.5)
ax.add_patch(rect)
ax.text(8.25, 4.15, 'ALL-REDUCE  (NCCL: sum partial results across 4 GPUs)', fontsize=10,
        ha='center', color='white', fontweight='bold')
for x in gpu_xs:
    ax.annotate('', xy=(x+0.75, 4.5), xytext=(x+0.75, 4.8),
                arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

# MLP (similar pattern)
for i, (x, color) in enumerate(zip(gpu_xs, gpu_colors)):
    rect = mpatches.FancyBboxPatch((x, 2.5), 1.5, 0.8, boxstyle="round,pad=0.05",
        facecolor=color, alpha=0.85, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+0.75, 2.9, f'MLP\n(partial)', fontsize=7, ha='center', color='white', fontweight='bold')
    ax.annotate('', xy=(x+0.75, 3.3), xytext=(x+0.75, 3.8),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.2))
ax.text(0.3, 2.9, 'Gate+Up\n(column)\nDown (row)', fontsize=7, fontweight='bold', color='#555')

# Second ALL-REDUCE
rect = mpatches.FancyBboxPatch((1.5, 1.5), 13.5, 0.7, boxstyle="round,pad=0.08",
    facecolor=RED, alpha=0.9, edgecolor='white', linewidth=2.5)
ax.add_patch(rect)
ax.text(8.25, 1.85, 'ALL-REDUCE  (sum MLP partial results)', fontsize=10,
        ha='center', color='white', fontweight='bold')
for x in gpu_xs:
    ax.annotate('', xy=(x+0.75, 2.2), xytext=(x+0.75, 2.5),
                arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

ax.text(8, 0.5, '2 All-Reduces per layer. Total: 2 x num_layers per forward pass.',
        fontsize=10, ha='center', style='italic', color='#555')

ax.set_title('Tensor Parallelism: One Transformer Layer Across 4 GPUs',
             fontsize=15, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

In [ ]:
class TensorParallelLayer:
    """
    Simulates a tensor-parallel transformer layer.
    
    Shows how weights are sharded and how all-reduce works.
    Source reference: vllm/model_executor/layers/linear.py
    """
    
    def __init__(self, hidden_size: int, intermediate_size: int, 
                 num_heads: int, head_dim: int, tp_size: int, rank: int):
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.tp_size = tp_size
        self.rank = rank
        
        # Attention: column parallel QKV, row parallel output
        self.heads_per_tp = num_heads // tp_size
        self.qkv_size_per_tp = self.heads_per_tp * head_dim * 3  # Q, K, V
        
        # MLP: column parallel gate+up, row parallel down
        self.intermediate_per_tp = intermediate_size // tp_size
        
        # Create sharded weights (on CPU for simulation)
        self.W_qkv = torch.randn(hidden_size, self.qkv_size_per_tp) / hidden_size**0.5
        self.W_o = torch.randn(self.heads_per_tp * head_dim, hidden_size) / hidden_size**0.5
        self.W_gate = torch.randn(hidden_size, self.intermediate_per_tp) / hidden_size**0.5
        self.W_up = torch.randn(hidden_size, self.intermediate_per_tp) / hidden_size**0.5
        self.W_down = torch.randn(self.intermediate_per_tp, hidden_size) / hidden_size**0.5
        
        self.comm_bytes = 0
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with simulated all-reduce."""
        batch_size = x.shape[0]
        
        # === Attention ===
        # Column parallel: QKV projection (local, no communication)
        qkv = x @ self.W_qkv  # [batch, qkv_size_per_tp]
        
        # Simplified attention (just matmul for demo)
        q_size = self.heads_per_tp * self.head_dim
        q = qkv[:, :q_size]
        attn_output = q  # Simplified
        
        # Row parallel: output projection (produces partial result)
        partial_attn = attn_output @ self.W_o  # [batch, hidden_size]
        
        # ★ ALL-REDUCE: sum partial results across TP group
        attn_out = partial_attn  # In real code: all_reduce(partial_attn)
        self.comm_bytes += batch_size * self.hidden_size * 2  # FP16
        
        # Residual
        x = x + attn_out
        
        # === MLP ===
        # Column parallel: gate and up projections
        gate = x @ self.W_gate  # [batch, intermediate_per_tp]
        up = x @ self.W_up
        
        # Fused SiLU + multiply (local)
        h = torch.nn.functional.silu(gate) * up
        
        # Row parallel: down projection
        partial_mlp = h @ self.W_down  # [batch, hidden_size]
        
        # ★ ALL-REDUCE
        mlp_out = partial_mlp  # In real code: all_reduce(partial_mlp)
        self.comm_bytes += batch_size * self.hidden_size * 2
        
        x = x + mlp_out
        return x

# Demo: TP=4 for LLaMA-70B-like model
tp_size = 4
hidden_size = 8192
intermediate_size = 28672
num_heads = 64
head_dim = 128

print(f"=== Tensor Parallel Layer (TP={tp_size}) ===")
print(f"Full model: hidden={hidden_size}, heads={num_heads}")
print(f"Per GPU: heads={num_heads//tp_size}, intermediate={intermediate_size//tp_size}")

# Create one layer shard
layer = TensorParallelLayer(
    hidden_size, intermediate_size, num_heads, head_dim, tp_size, rank=0)

# Weight sizes
total_params_per_tp = sum(p.numel() for p in [
    layer.W_qkv, layer.W_o, layer.W_gate, layer.W_up, layer.W_down])
total_params_full = total_params_per_tp * tp_size

print(f"\nWeight parameters per GPU per layer: {total_params_per_tp:,}")
print(f"Total parameters per layer: {total_params_full:,}")
print(f"Weight memory per GPU per layer: {total_params_per_tp * 2 / 1024 / 1024:.1f} MB (FP16)")

# Run forward
x = torch.randn(32, hidden_size)
out = layer.forward(x)
print(f"\nForward pass: input {x.shape} -> output {out.shape}")
print(f"Communication per layer: {layer.comm_bytes / 1024:.1f} KB")

In [ ]:
# Analyze all-reduce communication volume

def compute_tp_communication(
    tp_size: int,
    batch_size: int,
    hidden_size: int,
    num_layers: int,
    dtype_bytes: int = 2,
) -> dict:
    """
    Calculate total communication for a full forward pass with TP.
    
    Ring all-reduce cost: 2 * (n-1)/n * data_size
    Two all-reduces per layer (attention + MLP)
    """
    data_per_allreduce = batch_size * hidden_size * dtype_bytes
    
    # Ring all-reduce: each GPU sends (n-1)/n of data twice
    ring_factor = 2 * (tp_size - 1) / tp_size
    comm_per_allreduce = data_per_allreduce * ring_factor
    
    allreduces_per_layer = 2  # attention + MLP
    total_allreduces = allreduces_per_layer * num_layers
    total_comm = comm_per_allreduce * total_allreduces
    
    return {
        'data_per_allreduce_MB': data_per_allreduce / 1024 / 1024,
        'comm_per_allreduce_MB': comm_per_allreduce / 1024 / 1024,
        'total_allreduces': total_allreduces,
        'total_comm_GB': total_comm / 1024 / 1024 / 1024,
        'ring_factor': ring_factor,
    }

# Analyze for various configurations
configs = [
    ('LLaMA-7B, TP=2', 2, 32, 4096, 32),
    ('LLaMA-13B, TP=2', 2, 32, 5120, 40),
    ('LLaMA-70B, TP=4', 4, 32, 8192, 80),
    ('LLaMA-70B, TP=8', 8, 32, 8192, 80),
    ('LLaMA-405B, TP=8', 8, 32, 16384, 126),
]

print(f"{'Config':<25} {'All-reduces':>12} {'Per AR (MB)':>12} {'Total (GB)':>12}")
print("-" * 64)

for name, tp, bs, hidden, layers in configs:
    r = compute_tp_communication(tp, bs, hidden, layers)
    print(f"{name:<25} {r['total_allreduces']:>12} "
          f"{r['comm_per_allreduce_MB']:>11.2f} {r['total_comm_GB']:>11.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Figure 2: Pipeline Parallelism — Layers Split Across GPUs ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16); ax.set_ylim(0, 6); ax.axis('off')
fig.patch.set_facecolor('white')

pp_stages = [
    ('GPU 0\nLayers 0-7',   BLUE),
    ('GPU 1\nLayers 8-15',  GREEN),
    ('GPU 2\nLayers 16-23', ORANGE),
    ('GPU 3\nLayers 24-31', PURPLE),
]

# Stage boxes
bw, bh = 3.0, 2.5
for i, (label, color) in enumerate(pp_stages):
    x = 1.0 + i * 3.7
    rect = mpatches.FancyBboxPatch((x, 2.0), bw, bh, boxstyle="round,pad=0.15",
        facecolor=color, alpha=0.85, edgecolor='white', linewidth=2.5)
    ax.add_patch(rect)
    ax.text(x + bw/2, 3.25, label, ha='center', va='center', fontsize=10,
            fontweight='bold', color='white')

# Arrows between stages (activation transfer)
for i in range(len(pp_stages) - 1):
    sx = 1.0 + i * 3.7 + bw
    dx = 1.0 + (i+1) * 3.7
    ax.annotate('', xy=(dx, 3.25), xytext=(sx, 3.25),
                arrowprops=dict(arrowstyle='->', color=RED, lw=2.5))
    ax.text((sx + dx) / 2, 3.8, 'Send\nactivations', fontsize=7,
            ha='center', color=RED, fontweight='bold')

# Micro-batch flow indicators
for mb in range(3):
    for i in range(len(pp_stages)):
        x = 1.0 + i * 3.7 + 0.3 + mb * 0.9
        y = 2.2 + mb * 0.3
        circle = plt.Circle((x, y), 0.15, facecolor=['#FF6B6B', '#FFD93D', '#6BCB77'][mb],
                            edgecolor='white', linewidth=1)
        ax.add_patch(circle)
    if mb == 0:
        ax.text(15.5, 2.2, 'MB0', fontsize=8, color='#FF6B6B', fontweight='bold')
    elif mb == 1:
        ax.text(15.5, 2.5, 'MB1', fontsize=8, color='#FFD93D', fontweight='bold')
    else:
        ax.text(15.5, 2.8, 'MB2', fontsize=8, color='#6BCB77', fontweight='bold')

ax.text(8, 0.8, 'Each stage processes its layers, then sends activations to the next stage.\n'
        'Multiple micro-batches can be in-flight for better utilization.',
        ha='center', fontsize=9, style='italic', color='#555')

ax.set_title('Pipeline Parallelism: Layers Distributed Across GPUs with Micro-batch Flow',
             fontsize=14, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

## 5. Pipeline Parallelism (PP)

### How PP Works

```
Pipeline Parallelism (PP=4, 32 layers):

Stage 0 (GPU 0): Layers 0-7
Stage 1 (GPU 1): Layers 8-15
Stage 2 (GPU 2): Layers 16-23
Stage 3 (GPU 3): Layers 24-31

Forward pass (1 micro-batch):
Time →
Stage 0: [▓▓▓▓▓]──────────────────────────────
Stage 1:         [▓▓▓▓▓]──────────────────────
Stage 2:                 [▓▓▓▓▓]──────────────
Stage 3:                         [▓▓▓▓▓]──────
                  ↑        ↑        ↑
              Send activation between stages

Problem: Pipeline bubble (idle time)!
Efficiency = useful_compute / (useful_compute + bubble)
           ≈ (pp_size - 1) / pp_size × micro_batches
```

### Inference vs Training PP

For inference, PP is simpler than training (no backward pass):
- Each micro-batch flows through all stages sequentially
- Multiple micro-batches can be in-flight simultaneously
- Bubble overhead is less critical due to continuous batching

In [ ]:
def simulate_pipeline_schedule(
    pp_size: int,
    num_microbatches: int,
    stage_time: float = 1.0,  # Time per stage per micro-batch
    transfer_time: float = 0.1,  # Time to send activations
) -> dict:
    """
    Simulate pipeline parallel execution schedule.
    """
    # Schedule: when does each (stage, microbatch) start and end?
    start_times = {}  # (stage, mb) -> start_time
    end_times = {}    # (stage, mb) -> end_time
    
    for mb in range(num_microbatches):
        for stage in range(pp_size):
            # Can't start until:
            # 1. Previous stage sent its output (for this mb)
            # 2. This stage finished previous mb
            
            prev_stage_done = end_times.get((stage - 1, mb), 0) + transfer_time if stage > 0 else mb * stage_time
            prev_mb_done = end_times.get((stage, mb - 1), 0) if mb > 0 else 0
            
            start = max(prev_stage_done, prev_mb_done)
            end = start + stage_time
            
            start_times[(stage, mb)] = start
            end_times[(stage, mb)] = end
    
    total_time = max(end_times.values())
    ideal_time = num_microbatches * stage_time  # If no bubble
    bubble_time = total_time - ideal_time
    efficiency = ideal_time / total_time
    
    return {
        'total_time': total_time,
        'ideal_time': ideal_time,
        'bubble_time': bubble_time,
        'efficiency': efficiency,
        'start_times': start_times,
        'end_times': end_times,
    }

# Test
for pp in [2, 4, 8]:
    for mbs in [1, 4, 8, 16]:
        r = simulate_pipeline_schedule(pp, mbs)
        print(f"PP={pp}, MBs={mbs:>2}: total={r['total_time']:.1f}, "
              f"bubble={r['bubble_time']:.1f}, efficiency={r['efficiency']:.1%}")

In [ ]:
# Visualize pipeline schedule

def visualize_pipeline(pp_size: int, num_microbatches: int):
    result = simulate_pipeline_schedule(pp_size, num_microbatches)
    
    fig, ax = plt.subplots(figsize=(16, pp_size * 1.2 + 1))
    
    colors = plt.cm.Set3(np.linspace(0, 1, num_microbatches))
    
    for stage in range(pp_size):
        for mb in range(num_microbatches):
            start = result['start_times'][(stage, mb)]
            end = result['end_times'][(stage, mb)]
            
            rect = patches.FancyBboxPatch(
                (start, pp_size - stage - 1), end - start, 0.8,
                boxstyle="round,pad=0.02",
                facecolor=colors[mb], edgecolor='black', linewidth=0.5)
            ax.add_patch(rect)
            ax.text((start + end) / 2, pp_size - stage - 0.6,
                    f'MB{mb}', ha='center', va='center', fontsize=8)
    
    ax.set_xlim(-0.5, result['total_time'] + 0.5)
    ax.set_ylim(-0.5, pp_size + 0.5)
    ax.set_yticks(range(pp_size))
    ax.set_yticklabels([f'Stage {pp_size - i - 1}' for i in range(pp_size)])
    ax.set_xlabel('Time', fontsize=12)
    ax.set_title(f'Pipeline Schedule (PP={pp_size}, {num_microbatches} micro-batches, '
                 f'efficiency={result["efficiency"]:.0%})', fontsize=14)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(f'/tmp/pipeline_pp{pp_size}_mb{num_microbatches}.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_pipeline(4, 8)

## 6. NCCL Communication Primitives

### All-Reduce Implementation

```python
# From vllm/distributed/communication_op.py
def tensor_model_parallel_all_reduce(input_: torch.Tensor) -> torch.Tensor:
    """All-reduce across tensor parallel group."""
    if get_tensor_model_parallel_world_size() == 1:
        return input_
    
    # Use NCCL all-reduce
    torch.distributed.all_reduce(
        input_, 
        group=get_tensor_model_parallel_group()
    )
    return input_
```

### Communication Bandwidth

```
Interconnect Bandwidths:
  NVLink 3.0 (A100): 600 GB/s total (12 links × 50 GB/s)
  NVLink 4.0 (H100): 900 GB/s total (18 links × 50 GB/s)
  PCIe 4.0: ~32 GB/s
  InfiniBand HDR: 200 Gbps = 25 GB/s
  InfiniBand NDR: 400 Gbps = 50 GB/s

Ring All-reduce Time:
  T = 2 * (n-1)/n * D / BW
  
  Example: 8 GPUs, 16MB data, NVLink 600 GB/s
  T = 2 * 7/8 * 16MB / 600 GB/s = 47 μs
```

In [ ]:
def estimate_allreduce_time(
    data_size_bytes: int,
    num_gpus: int,
    bandwidth_gbps: float,  # GB/s per link
    latency_us: float = 5.0,  # Base latency in microseconds
) -> float:
    """
    Estimate ring all-reduce time.
    
    Ring all-reduce: 2 * (n-1) / n * data / bandwidth
    Plus base latency × 2 × (n-1) (two phases, n-1 steps each)
    """
    # Ring algorithm: data split into n chunks, sent n-1 times
    ring_factor = 2 * (num_gpus - 1) / num_gpus
    transfer_time_us = ring_factor * data_size_bytes / (bandwidth_gbps * 1e3)  # us
    total_latency_us = latency_us * 2 * (num_gpus - 1)
    
    return transfer_time_us + total_latency_us

# Analyze communication time for different interconnects
data_sizes = [2**i for i in range(10, 26)]  # 1KB to 32MB

interconnects = {
    'NVLink 3.0 (A100)': 600,
    'NVLink 4.0 (H100)': 900,
    'PCIe 4.0': 32,
    'InfiniBand HDR': 25,
}

fig, ax = plt.subplots(figsize=(12, 6))

for name, bw in interconnects.items():
    times = [estimate_allreduce_time(ds, 8, bw) for ds in data_sizes]
    ax.plot([ds/1024 for ds in data_sizes], times, '-o', label=name, markersize=3)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Data Size (KB)', fontsize=12)
ax.set_ylabel('All-Reduce Time (us)', fontsize=12)
ax.set_title('All-Reduce Latency vs Data Size (8 GPUs)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/allreduce_latency.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. How vLLM Shards Model Weights

### Source: `vllm/model_executor/layers/linear.py`

```python
class ColumnParallelLinear(LinearBase):
    """Linear layer with column parallelism.
    
    The linear layer is defined as Y = XA + b. A is parallelized along
    its second dimension as A = [A_1, ..., A_p].
    """
    def __init__(self, input_size, output_size, ...):
        self.tp_size = get_tensor_model_parallel_world_size()
        self.output_size_per_partition = output_size // self.tp_size
        # Weight: [input_size, output_size_per_partition]
        self.weight = Parameter(
            torch.empty(self.output_size_per_partition, input_size))

class RowParallelLinear(LinearBase):
    """Linear layer with row parallelism.
    
    A is parallelized along its first dimension:
    A = [A_1; A_2; ...; A_p]
    Y = XA + b = X[A_1; A_2; ...]  + b
    
    Requires all-reduce to combine partial results.
    """
    def __init__(self, input_size, output_size, ...):
        self.tp_size = get_tensor_model_parallel_world_size()
        self.input_size_per_partition = input_size // self.tp_size
        # Weight: [output_size, input_size_per_partition]
        self.weight = Parameter(
            torch.empty(output_size, self.input_size_per_partition))
```

In [ ]:
def analyze_weight_sharding(model_config: dict, tp_size: int):
    """
    Analyze how model weights are sharded across GPUs.
    """
    hidden = model_config['hidden_size']
    intermediate = model_config['intermediate_size']
    num_heads = model_config['num_attention_heads']
    num_kv_heads = model_config.get('num_kv_heads', num_heads)
    head_dim = hidden // num_heads
    num_layers = model_config['num_layers']
    vocab_size = model_config['vocab_size']
    
    # Per-layer weights
    per_layer = {
        'W_q': (hidden, num_heads * head_dim // tp_size),
        'W_k': (hidden, num_kv_heads * head_dim // tp_size),
        'W_v': (hidden, num_kv_heads * head_dim // tp_size),
        'W_o': (num_heads * head_dim // tp_size, hidden),
        'W_gate': (hidden, intermediate // tp_size),
        'W_up': (hidden, intermediate // tp_size),
        'W_down': (intermediate // tp_size, hidden),
        'input_layernorm': (hidden,),
        'post_attn_layernorm': (hidden,),
    }
    
    # Non-layer weights
    non_layer = {
        'embedding': (vocab_size, hidden),  # Not sharded in TP
        'lm_head': (hidden, vocab_size // tp_size),  # Column parallel
        'final_norm': (hidden,),
    }
    
    layer_params = sum(np.prod(shape) for shape in per_layer.values())
    non_layer_params = sum(np.prod(shape) for shape in non_layer.values())
    total_params = layer_params * num_layers + non_layer_params
    total_memory_gb = total_params * 2 / 1024**3  # FP16
    
    print(f"\n=== Weight Sharding Analysis (TP={tp_size}) ===")
    print(f"\nPer-layer weights per GPU:")
    for name, shape in per_layer.items():
        params = np.prod(shape)
        print(f"  {name:<25} {str(shape):<20} {params:>12,} params")
    print(f"  {'TOTAL per layer':<25} {'':20} {layer_params:>12,} params")
    
    print(f"\nNon-layer weights per GPU:")
    for name, shape in non_layer.items():
        params = np.prod(shape)
        print(f"  {name:<25} {str(shape):<20} {params:>12,} params")
    
    print(f"\nTotal per GPU: {total_params:,} params = {total_memory_gb:.2f} GB (FP16)")
    print(f"Total model: {total_params * tp_size:,} params = {total_memory_gb * tp_size:.2f} GB (FP16)")

# LLaMA-70B
llama_70b = {
    'hidden_size': 8192,
    'intermediate_size': 28672,
    'num_attention_heads': 64,
    'num_kv_heads': 8,
    'num_layers': 80,
    'vocab_size': 32000,
}

for tp in [1, 2, 4, 8]:
    analyze_weight_sharding(llama_70b, tp)

## 8. Multi-Node Configuration

### Setup and Configuration

```bash
# Single node, 8 GPUs, TP=8
vllm serve meta-llama/Llama-2-70b-hf \
    --tensor-parallel-size 8

# Two nodes, 8 GPUs each, TP=8, PP=2
# Node 0 (head):
vllm serve meta-llama/Llama-2-70b-hf \
    --tensor-parallel-size 8 \
    --pipeline-parallel-size 2

# Multi-node requires Ray:
# Node 0: ray start --head
# Node 1: ray start --address=<head_ip>:6379
# Then run vllm on node 0
```

### Best Practices

```
TP within node (NVLink):
  - 600+ GB/s bandwidth
  - Low latency
  - Support TP=2,4,8

PP across nodes (InfiniBand):
  - 25-50 GB/s bandwidth
  - Higher latency
  - Only sends activations (small)

Rule of thumb:
  TP = number of GPUs per node (up to 8)
  PP = number of nodes
```

In [ ]:
# Compare communication costs: TP within node vs PP across nodes

def compare_tp_pp_configs(
    total_gpus: int,
    batch_size: int,
    hidden_size: int,
    num_layers: int,
):
    """
    Compare different TP/PP configurations for a given GPU count.
    """
    configs = []
    for tp in [1, 2, 4, 8]:
        pp = total_gpus // tp
        if pp < 1 or tp * pp != total_gpus:
            continue
        
        # TP communication: 2 all-reduces per layer × layers_per_stage
        layers_per_stage = num_layers // pp
        tp_data_per_ar = batch_size * hidden_size * 2  # FP16
        tp_ring_factor = 2 * (tp - 1) / max(tp, 1)
        tp_comm_per_gpu = tp_ring_factor * tp_data_per_ar * 2 * layers_per_stage  # 2 ARs per layer
        
        # PP communication: send activations between stages
        pp_data_per_transfer = batch_size * hidden_size * 2  # FP16
        pp_comm = pp_data_per_transfer * (pp - 1)  # pp-1 transfers
        
        # Estimate time (NVLink for TP, InfiniBand for PP)
        nvlink_bw = 600e9  # bytes/s
        ib_bw = 25e9  # bytes/s
        
        tp_time_ms = tp_comm_per_gpu / nvlink_bw * 1000 if tp > 1 else 0
        pp_time_ms = pp_comm / ib_bw * 1000 if pp > 1 else 0
        
        # Pipeline bubble overhead
        stage_compute_ms = layers_per_stage * 0.5  # Rough estimate
        bubble_ms = (pp - 1) * stage_compute_ms
        
        configs.append({
            'tp': tp,
            'pp': pp,
            'tp_comm_MB': tp_comm_per_gpu / 1024 / 1024,
            'pp_comm_MB': pp_comm / 1024 / 1024,
            'tp_time_ms': tp_time_ms,
            'pp_time_ms': pp_time_ms,
            'bubble_ms': bubble_ms,
            'total_overhead_ms': tp_time_ms + pp_time_ms + bubble_ms,
        })
    
    return configs

# 8 GPUs, LLaMA-70B
configs = compare_tp_pp_configs(8, 32, 8192, 80)

print(f"{'TP':>3} {'PP':>3} {'TP Comm':>10} {'PP Comm':>10} {'TP Time':>10} {'PP Time':>10} {'Bubble':>10} {'Total':>10}")
print("-" * 70)
for c in configs:
    print(f"{c['tp']:>3} {c['pp']:>3} {c['tp_comm_MB']:>9.1f}M {c['pp_comm_MB']:>9.1f}M "
          f"{c['tp_time_ms']:>9.2f}ms {c['pp_time_ms']:>9.2f}ms {c['bubble_ms']:>9.1f}ms "
          f"{c['total_overhead_ms']:>9.2f}ms")

## 9. Distributed Worker Source Code

### Worker Initialization

```python
# From vllm/worker/worker.py (simplified)
class Worker:
    def __init__(self, model_config, parallel_config, ...):
        self.rank = parallel_config.rank
        self.local_rank = parallel_config.local_rank
        self.tp_size = parallel_config.tensor_parallel_size
        self.pp_size = parallel_config.pipeline_parallel_size
    
    def init_device(self):
        # Set CUDA device for this worker
        torch.cuda.set_device(self.local_rank)
        
        # Initialize distributed process group
        init_distributed_environment(
            world_size=self.tp_size * self.pp_size,
            rank=self.rank,
            distributed_init_method=self.distributed_init_method,
        )
        
        # Initialize model parallel groups
        ensure_model_parallel_initialized(
            tensor_model_parallel_size=self.tp_size,
            pipeline_model_parallel_size=self.pp_size,
        )
    
    def load_model(self):
        # Load model weights (each GPU loads its shard)
        self.model_runner = ModelRunner(
            model_config=self.model_config,
            parallel_config=self.parallel_config,
        )
        self.model_runner.load_model()
    
    def execute_model(self, execute_model_req):
        # Run forward pass on this GPU's model shard
        output = self.model_runner.execute_model(execute_model_req)
        return output
```

## 10. Communication Overhead Profiling

In [ ]:
def profile_communication_overhead(
    model_configs: List[dict],
    tp_sizes: List[int],
    batch_size: int = 32,
):
    """
    Profile communication overhead as percentage of total execution time.
    """
    results = []
    
    for model_cfg in model_configs:
        hidden = model_cfg['hidden_size']
        layers = model_cfg['num_layers']
        name = model_cfg['name']
        
        for tp in tp_sizes:
            if tp > 8:
                continue
            
            # Compute time (rough estimate)
            # FLOPs per layer per token (decode): ~12 * hidden^2
            flops_per_layer = 12 * hidden * hidden * batch_size
            total_flops = flops_per_layer * layers / tp  # Divided by TP
            
            # A100: ~312 TFLOPS FP16
            gpu_tflops = 312e12
            compute_time_ms = total_flops / gpu_tflops * 1000
            
            # Communication time
            ar_data = batch_size * hidden * 2  # FP16
            ring_factor = 2 * (tp - 1) / max(tp, 1)
            ar_time_per_layer = ring_factor * ar_data / 600e9 * 1000  # NVLink
            comm_time_ms = ar_time_per_layer * 2 * layers  # 2 ARs per layer
            
            total_time = compute_time_ms + comm_time_ms
            comm_fraction = comm_time_ms / total_time if total_time > 0 else 0
            
            results.append({
                'model': name,
                'tp': tp,
                'compute_ms': compute_time_ms,
                'comm_ms': comm_time_ms,
                'total_ms': total_time,
                'comm_fraction': comm_fraction,
            })
    
    return results

models = [
    {'name': 'LLaMA-7B', 'hidden_size': 4096, 'num_layers': 32},
    {'name': 'LLaMA-13B', 'hidden_size': 5120, 'num_layers': 40},
    {'name': 'LLaMA-70B', 'hidden_size': 8192, 'num_layers': 80},
]

results = profile_communication_overhead(models, [1, 2, 4, 8])

print(f"{'Model':<12} {'TP':>3} {'Compute':>10} {'Comm':>10} {'Total':>10} {'Comm %':>8}")
print("-" * 56)
for r in results:
    print(f"{r['model']:<12} {r['tp']:>3} {r['compute_ms']:>9.2f}ms "
          f"{r['comm_ms']:>9.3f}ms {r['total_ms']:>9.2f}ms {r['comm_fraction']:>7.1%}")

In [ ]:
# Visualize communication overhead
fig, ax = plt.subplots(figsize=(12, 6))

model_names = list(set(r['model'] for r in results))
tp_sizes_used = sorted(set(r['tp'] for r in results))

x = np.arange(len(model_names))
width = 0.2

for i, tp in enumerate(tp_sizes_used):
    fracs = []
    for model in model_names:
        matching = [r for r in results if r['model'] == model and r['tp'] == tp]
        if matching:
            fracs.append(matching[0]['comm_fraction'] * 100)
        else:
            fracs.append(0)
    
    offset = (i - len(tp_sizes_used)/2 + 0.5) * width
    bars = ax.bar(x + offset, fracs, width, label=f'TP={tp}', alpha=0.8)

ax.set_ylabel('Communication Overhead (%)', fontsize=12)
ax.set_title('Communication Overhead vs Model Size and TP', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/comm_overhead.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary

### Key Decisions

| Scenario | Recommended Config | Reason |
|----------|-------------------|--------|
| 7B on 1 GPU | TP=1, PP=1 | Fits in single GPU |
| 70B on 2 GPUs | TP=2 | NVLink within node |
| 70B on 8 GPUs | TP=8 | Maximum throughput |
| 405B on 2 nodes | TP=8, PP=2 | TP within node, PP across |
| Any model, max throughput | Add DP replicas | Independent serving instances |

### Communication Hierarchy
1. **Within node** (NVLink): Use for TP (high bandwidth)
2. **Across nodes** (InfiniBand): Use for PP (low data volume)
3. **Data parallelism**: No communication needed (replicate model)

## Exercises

### Exercise 1: Optimal TP/PP Split
Given 16 GPUs across 2 nodes (8 per node), find the optimal TP/PP split for serving LLaMA-405B. Consider NVLink bandwidth within nodes and InfiniBand across nodes.

### Exercise 2: Communication-Computation Overlap
Design a scheme that overlaps all-reduce communication with computation. Which operations in the transformer layer can be overlapped with the previous layer's all-reduce?

### Exercise 3: Profile Communication
Write a script that measures actual NCCL all-reduce bandwidth on your system. Compare with theoretical peak.

### Exercise 4: Expert Parallelism
Design a parallelism strategy for a Mixture-of-Experts (MoE) model like Mixtral-8x7B. How would you shard the experts across GPUs?

In [ ]:
# Exercise 1 Starter

print("=== Exercise 1: Optimal TP/PP Split for 16 GPUs ===")
print("\nLLaMA-405B: 126 layers, hidden=16384")
print("2 nodes × 8 GPUs each")

for tp in [1, 2, 4, 8, 16]:
    pp = 16 // tp
    if tp * pp != 16:
        continue
    
    # Check if TP fits within a node
    tp_within_node = tp <= 8
    
    configs = compare_tp_pp_configs(16, 32, 16384, 126)
    matching = [c for c in configs if c['tp'] == tp]
    
    if matching:
        c = matching[0]
        feasibility = 'Good' if tp_within_node else 'Bad (TP across nodes!)'
        print(f"  TP={tp:>2}, PP={pp:>2}: total overhead={c['total_overhead_ms']:.2f}ms "
              f"[{feasibility}]")